# Desafio Lighthouse 2026.2 — LH Nautical

Notebook único cobrindo as 7 questões do desafio, seguindo a mesma organização usada na resolução do 2026.1: uma seção markdown por questão, com o raciocínio explicado antes e depois de cada bloco de código — não só o resultado.

**Organização do código (fase de refinamento):** enquanto as respostas ainda estão sendo ajustadas, o código de cada questão mora só em `../Submissão/Q1/`, `../Submissão/Q2/` etc. — os mesmos arquivos que vão pro formulário. Este notebook carrega e executa esses arquivos, narra o raciocínio em markdown. Evita manter duas cópias divergentes (`src/` e `Submissão/`) enquanto a resposta ainda pode mudar; ao final do desafio, os arquivos definitivos de `Submissão/` são copiados para `Workspace/src/` como código de referência do repositório.

**Engine:** DuckDB lendo os CSVs brutos de `../data/raw/1-lh_nautical_csv/` diretamente. A modelagem final em PostgreSQL (`schema.sql`) é construída nas Questões 2 e 3; até lá, o DuckDB serve só para exploração ad hoc sobre o CSV cru, sem inventar tipos definitivos.

**Decisão de robustez importante (detalhada em `../../Anotações/comentarios.md`):** todo `read_csv_auto` deste notebook usa `sample_size=-1` (varredura completa do arquivo para inferir tipos), não o padrão de 20.480 linhas do DuckDB. Confirmei com um teste reproduzível que a amostra padrão pode inferir um tipo errado e quebrar em runtime quando um valor fora do padrão da coluna cai fora da janela amostrada — e `orders.csv` (48.998 linhas) já ultrapassa esse padrão. Como os CSVs deste desafio são pequenos o suficiente (o maior tem 147 mil linhas), a varredura completa custa segundos — não há razão para arriscar.

## Q1 — EDA

### Setup — carregar `orders` no DuckDB

Premissas obrigatórias da Q1: usar apenas a tabela `orders`, sem limpeza/tratamento. `read_csv_auto` só faz inferência de tipo para viabilizar a leitura (necessário para ter uma tabela SQL de verdade) — não corrige, filtra ou descarta nada. `sample_size=-1` garante que essa inferência olhe as 48.998 linhas, não uma amostra.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE VIEW orders AS
    SELECT * FROM read_csv_auto('../data/raw/1-lh_nautical_csv/orders.csv', sample_size=-1)
""")

con.sql("DESCRIBE orders")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_number    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ channel         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ salesperson_id  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ location_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ status          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ subtotal        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ discount_amount │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ total           │ DOUBLE      │ YES 

### Q1.1 — SQL (Parte 1: visão geral + Parte 2: valores numéricos)

Query em `../Submissão/Q1/1.1.sql` — o mesmo arquivo que vai pro upload da 1.1. Cobre os 5 itens pedidos (linhas, datas min/max, total min/max/médio) mais a contagem de colunas via `information_schema.columns` — pedida na Parte 1 do enunciado principal. Contar colunas assim, e não com `COUNT(*)`, é uma lição direta da revisão do 2026.1: `COUNT(*)` conta linhas, nunca colunas.

In [2]:
with open("../Submissão/Q1/1.1.sql") as f:
    query = f.read()

con.sql(query)

┌──────────────┬───────────────┬─────────────────────┬─────────────────────┬───────────┬───────────┬────────────────────┐
│ total_linhas │ total_colunas │      data_min       │      data_max       │ total_min │ total_max │    total_media     │
│    int64     │     int64     │      timestamp      │      timestamp      │  double   │  double   │       double       │
├──────────────┼───────────────┼─────────────────────┼─────────────────────┼───────────┼───────────┼────────────────────┤
│        48998 │            13 │ 2020-01-01 01:19:28 │ 2026-12-31 23:43:09 │     32.62 │ 127262.02 │ 28704.992077227675 │
└──────────────┴───────────────┴─────────────────────┴─────────────────────┴───────────┴───────────┴────────────────────┘

**Resultado (Parte 1 + Parte 2):**

- Quantidade total de linhas: **48.998**
- Quantidade total de colunas: **13**
- Intervalo de datas (`created_at`): **2020-01-01 01:19:28** a **2026-12-31 23:43:09**
- `total`: mínimo **R$ 32,62** · máximo **R$ 127.262,02** · médio **R$ 28.704,99**

### Q1.2 — Validação

**Qual é o valor médio registrado na coluna "total"?**

R$ 28.704,99

### Investigação de qualidade de dados (apoio à Q1.3)

Não faz parte do código pedido em 1.1 — sustenta o diagnóstico da Parte 3. Cada célula abaixo carrega uma pergunta isolada de `../Submissão/Q1/dq_*.sql`: nulos por coluna e consistência aritmética, se o nulo em `salesperson_id` é estrutural ou falta de dado, distribuição de `status`, outliers em `total` via IQR, e cobertura de datas por ano.

In [3]:
with open("../Submissão/Q1/dq_nulls_and_consistency.sql") as f:
    query = f.read()

con.sql(query)

┌────────────┬──────────────────┬─────────────────────┬──────────────────┬──────────────┬─────────────┬────────────────────┬──────────────────────────────────┬────────────────┬─────────────────────────┐
│ null_total │ null_customer_id │ null_salesperson_id │ null_location_id │ null_channel │ null_status │ total_nao_positivo │ total_inconsistente_com_subtotal │ ids_duplicados │ order_number_duplicados │
│   int64    │      int64       │        int64        │      int64       │    int64     │    int64    │       int64        │              int64               │     int64      │          int64          │
├────────────┼──────────────────┼─────────────────────┼──────────────────┼──────────────┼─────────────┼────────────────────┼──────────────────────────────────┼────────────────┼─────────────────────────┤
│          0 │                0 │               24131 │                0 │            0 │           0 │                  0 │                                0 │              0 │            

In [4]:
with open("../Submissão/Q1/dq_salesperson_null_by_channel.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬─────────┬──────────────┐
│  channel  │ pedidos │ sem_vendedor │
│  varchar  │  int64  │    int64     │
├───────────┼─────────┼──────────────┤
│ ecommerce │   34342 │        24131 │
│ pos       │   14656 │            0 │
└───────────┴─────────┴──────────────┘

In [5]:
with open("../Submissão/Q1/dq_status_distribution.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ paid      │ 34365 │
│ confirmed │  7335 │
│ cancelled │  4847 │
│ draft     │  2451 │
└───────────┴───────┘

In [6]:
with open("../Submissão/Q1/dq_outliers_iqr.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬────────────┬────────────────┬─────────────────┐
│    q1     │     q3     │ outliers_acima │ outliers_abaixo │
│  double   │   double   │     int64      │      int64      │
├───────────┼────────────┼────────────────┼─────────────────┤
│ 13171.235 │ 40941.8825 │            452 │               0 │
└───────────┴────────────┴────────────────┴─────────────────┘

In [7]:
with open("../Submissão/Q1/dq_orders_by_year.sql") as f:
    query = f.read()

con.sql(query)

┌───────┬───────┐
│  ano  │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  2020 │  4466 │
│  2021 │  5088 │
│  2022 │  5856 │
│  2023 │  6697 │
│  2024 │  7666 │
│  2025 │  8957 │
│  2026 │ 10268 │
└───────┴───────┘

### Q1.3 — Interpretação (diagnóstico de confiabilidade)

Isoladamente, a tabela `orders` está limpa e internamente consistente: não há valores nulos em `id`, `customer_id`, `location_id`, `channel`, `status`, `total` ou nas colunas de data; `total` nunca é nulo, zero ou negativo; `subtotal - discount_amount` bate com `total` nas 48.998 linhas (0 divergências); e não há `id` nem `order_number` duplicados. O único `NULL` relevante é `salesperson_id`, ausente em 24.131 das 34.342 vendas do canal `ecommerce` e presente em 0% das vendas `pos`. **Correção sobre uma leitura anterior deste diagnóstico:** os nulos ocorrerem só no `ecommerce` não significa que todo pedido `ecommerce` careça de vendedor — 10.211 dos 34.342 pedidos `ecommerce` (~30%) têm `salesperson_id` preenchido. Ou seja, há uma correlação forte (100% dos nulos vêm do canal online), mas não uma regra determinística ("pedido online nunca tem vendedor"). Sem uma regra de negócio explícita (ex.: vendedor só é registrado em venda assistida/por telefone dentro do canal ecommerce), trato essa ausência como um padrão a investigar nas próximas questões, não como uma explicação estrutural fechada — e mantenho em aberto se isso é ou não um problema de qualidade.

**Outliers em `total`:** a distribuição vai de R$32,62 a R$127.262,02, com média (R$28.704,99) puxada acima da mediana (R$25.917,84) por uma cauda de valores altos. Pelo critério de IQR (1,5×), 452 pedidos (~0,92%) ficam acima do limite superior (~R$99.517), e nenhum abaixo do limite inferior. Dado o contexto — varejo náutico, motores de popa e embarcações — pedidos de dezenas de milhares de reais são plausíveis, não parecem erro de digitação (sem negativos, sem sinal de vírgula/ponto trocado). Mas isso não é uma confirmação: só com `orders` não dá pra validar se são de fato pedidos legítimos de grande porte — a legitimidade só pode ser confirmada relacionando esses pedidos com `order_items`, produtos e preços unitários, o que exige o schema completo (Q2/Q3). Recomendação: não descartar automaticamente, mas tratar como "plausível, ainda não validado".

**Ponto a registrar, não necessariamente um erro:** `created_at` vai até 2026-12-31. Contando a partir de hoje (2026-08-10, inclusive), são 4.338 pedidos (~9%) com data igual ou posterior a hoje; estritamente depois de hoje, 4.322 — a diferença são 16 pedidos do próprio dia 10/08. O enunciado avisa que a base é fictícia e cobre 2020–2026, então trato isso como dado sintético gerado para o período todo, não como bug de carga; numa base real isso exigiria checar se são pedidos futuros legítimos (pré-venda) ou erro de ingestão.

**Veredito:** como tabela isolada, `orders` está pronta para as agregações simples pedidas aqui (contagens, min/max/média) — não há tratamento prévio necessário para isso. Não está pronta, porém, para sustentar sozinha as perguntas de negócio das próximas questões: (a) ainda não é possível confirmar que as demais 23 tabelas têm a mesma consistência — em particular `order_items`, cuja soma por pedido deveria reconciliar com `orders.total`, só será verificável depois que Q2/Q3 carregarem o schema completo; (b) pedidos com status `cancelled` (4.847, ~10%) e `draft` (2.451, ~5%) provavelmente não deveriam entrar em métricas de faturamento realizado — esse filtro não foi pedido aqui (a instrução foi "não trate os dados"), mas será necessário nas questões de análise de vendas/clientes à frente.